In [1]:
### working with files and folders
import os
import pathlib
import shutil
from glob import glob
import time
import zipfile

### keeping passwords out of the code
from getpass import getpass

### GBIF packages
import pygbif.occurrences as occ
import pygbif.species as species

### tabular and spatial data
import pandas as pd
import geopandas as gpd

### opening raster files (land cover maps)
import rioxarray as rxr

### interactive plots
import hvplot.pandas

### interactive maps of raster data (land cover maps)
import hvplot.xarray

In [2]:
### change this to reset = True if you need to type your credentials again
reset = False

### ask for GBIF credentials and keep them only for this session
### (never put credentials in the code)
if (not ('GBIF_USER' in os.environ)) or reset:
    os.environ['GBIF_USER'] = input('GBIF username:')
if (not ('GBIF_PWD' in os.environ)) or reset:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
if (not ('GBIF_EMAIL' in os.environ)) or reset:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [3]:
### make a folder for the GBIF data
data_dir = os.path.join(
    pathlib.Path.home(), 'earth-analytics', 'data', 'saemangeum-shorebirds')
gbif_dir = os.path.join(data_dir, 'gbif')
os.makedirs(gbif_dir, exist_ok = True)

### folder with the land cover maps from the Korean Ministry of Environment
landcover_dir = os.path.join(data_dir, 'landcover')

### find the GBIF key for the sandpiper family (Scolopacidae)
### (Scolopacidae is an accepted name, so the key is under 'usage', not 'acceptedUsage')
backbone = species.name_backbone(scientificName = 'Scolopacidae')
family_key = backbone['usage']['key']
family_key

'5282'

In [4]:
### set file name for download
gbif_pattern = os.path.join(gbif_dir, '*.csv')

### only download the data the first time you run this chunk (since it can be slow)
if not glob(gbif_pattern):

    ### only submit one request
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:

        ### submit a request to GBIF
        gbif_query = occ.download([

            ### all sandpipers and their relatives (family Scolopacidae)
            f"taxonKey = {family_key}",

            ### only records in South Korea
            'country = KR',

            ### filter out results without geographic coordinates
            'hasCoordinate = True',

            ### years before and after the seawall was closed in 2006
            'year >= 2000',
            'year <= 2025',
        ])

        ### grab the first result
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]

    ### wait for the download to build
    download_key = os.environ['GBIF_DOWNLOAD_KEY']
    wait = occ.download_meta(download_key)['status']
    while not wait == 'SUCCEEDED':
        wait = occ.download_meta(download_key)['status']
        time.sleep(5)

    ### download GBIF data when it's ready
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = gbif_dir)

    ### unzip GBIF data using the zipfile package
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_dir)

### find the extracted .csv file path (take the first result)
gbif_path = glob(gbif_pattern)[0]
gbif_path

'C:\\Users\\chwj1\\earth-analytics\\data\\saemangeum-shorebirds\\gbif\\0011495-261118162104854.csv'

In [5]:
### join land cover map sheets side by side
from rioxarray.merge import merge_arrays



In [6]:
### Level 1 land cover classes in the Korean land cover map
landcover_classes = {
    1: 'Urban',
    2: 'Agriculture',
    3: 'Forest',
    4: 'Grassland',
    5: 'Wetland',
    6: 'Barren',
    7: 'Water'}

### folder name of each map version -> the year the map shows
landcover_years = {'1990': '2001', '2000': '2010', '2010': '2019'}

### open the Saemangeum boundary in the land cover map's coordinate system
saemangeum_gdf = gpd.read_file('../data/saemangeum.geojson').to_crs(epsg = 5186)

### calculate the area of each land cover class inside the seawall for each year
area_list = []
for folder, year in landcover_years.items():

    ### open the merged land cover map for this year
    ### (mask_and_scale = True turns empty pixels into NaN, so clipping works safely)
    landcover_da = rxr.open_rasterio(
        os.path.join(landcover_dir, folder, f'combined_{folder}.tif'),
        mask_and_scale = True).squeeze()

    ### keep only the pixels inside the Saemangeum boundary
    saemangeum_da = landcover_da.rio.clip(saemangeum_gdf.geometry)

    ### count the pixels of each class (only codes 1 to 7, not empty pixels)
    pixel_counts = (
        pd.Series(saemangeum_da.values.ravel())
        .value_counts()
        .reindex(list(landcover_classes))
        .fillna(0))

    ### each pixel is 30 m x 30 m = 900 m2 = 0.0009 km2
    area_list.append((pixel_counts * 0.0009).rename(year))

### put the three years side by side and use class names
landcover_area_df = pd.concat(area_list, axis = 1).rename(index = landcover_classes)
landcover_area_df

c:\Users\chwj1\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


,2001,2010,2019
Urban,2.0394,3.7953,4.9932
Agriculture,1.0746,0.9882,10.7550
Forest,1.5570,1.6011,1.5300
Grassland,0.8397,0.8811,3.2796
Wetland,18.3132,18.2259,48.3093
Barren,1.9557,3.0879,19.2987
Water,250.5654,247.7655,188.1792


In [7]:
### official colors of the Level 1 land cover map (codes 1 to 7, in order)
landcover_colors = [
    '#e41a1c',  ### 1 Urban: red
    '#ffff00',  ### 2 Agriculture: yellow
    '#2d4f2d',  ### 3 Forest: dark green
    '#4caf50',  ### 4 Grassland: light green
    '#7b2d8b',  ### 5 Wetland: purple
    '#5ccfd6',  ### 6 Barren: light blue
    '#0000ff']  ### 7 Water: blue

### rectangle around Saemangeum for the maps
saemangeum_box_gdf = gpd.GeoDataFrame(geometry = saemangeum_gdf.envelope)

### make one land cover map for each year
hvplot.extension('bokeh')
landcover_plots = {}
for folder, year in landcover_years.items():

    ### open the land cover map and keep only the rectangle around Saemangeum
    landcover_da = rxr.open_rasterio(
        os.path.join(landcover_dir, folder, f'combined_{folder}.tif'),
        mask_and_scale = True).squeeze()
    box_da = landcover_da.rio.clip(saemangeum_box_gdf.geometry)

    ### map the land cover classes with the official colors
    landcover_plots[year] = (
        box_da.hvplot(
            x = 'x', y = 'y',
            cmap = landcover_colors,

            ### each class (1 to 7) gets one color
            clim = (0.5, 7.5),

            ### show the color bar only on the last map
            colorbar = (year == '2019'),

            ### keep the real shape of the area and hide the axes
            data_aspect = 1, frame_width = 200,
            xaxis = None, yaxis = None,
            title = f'Land cover in {year}',

            ### no hover box (the color bar already shows the class names)
            hover = False)

        ### put the class names on the color bar instead of the codes
        .opts(cticks = list(landcover_classes.items()))
        *

        ### draw the area inside the seawall on top
        saemangeum_gdf.hvplot(fill_color = None, line_color = 'black')
    )

### put the three maps side by side
landcover_maps = landcover_plots['2001'] + landcover_plots['2010'] + landcover_plots['2019']
landcover_maps

:Layout
   .Overlay.I   :Overlay
      .Image.I    :Image   [x,y]   (value)
      .Polygons.I :Polygons   [x,y]
   .Overlay.II  :Overlay
      .Image.I    :Image   [x,y]   (value)
      .Polygons.I :Polygons   [x,y]
   .Overlay.III :Overlay
      .Image.I    :Image   [x,y]   (value)
      .Polygons.I :Polygons   [x,y]

In [8]:
### Figure 2: land cover area inside the seawall
hvplot.extension('bokeh')
landcover_bar = landcover_area_df.drop(index = 'Water').hvplot.bar(
    title = 'Land cover inside the Saemangeum seawall (except water)',
    xlabel = 'Land cover class',
    ylabel = 'Area (km²)',
    frame_width = 500,
    legend = 'top_left',
    group_label = 'Year').opts(multi_level = False)
landcover_bar

:Bars   [index,Year]   (value)

In [9]:
### load the GBIF data (GBIF downloads are tab-delimited)
gbif_df = pd.read_csv(
    gbif_path,
    delimiter = '\t',
    index_col = 'gbifID',
    ### keep only the ID, location, and year columns
    usecols = ['gbifID', 'decimalLatitude', 'decimalLongitude', 'year'])

### convert to GeoDataFrame
gbif_gdf = gpd.GeoDataFrame(
    gbif_df,

    ### points from the lon/lat of each record
    geometry = gpd.points_from_xy(
        gbif_df.decimalLongitude,
        gbif_df.decimalLatitude),

    ### lat/lon coordinate system
    crs = 'EPSG:4326')

### number of records in each year
gbif_gdf.groupby('year').size()

year
2000       75
2001      119
2002      253
2003      222
2004      334
2005      156
2006      343
2007      376
2008      218
2009      269
2010      164
2011      128
2012      291
2013     1112
2014      520
2015      193
2016      997
2017     1803
2018     2108
2019     2571
2020     3588
2021     4057
2022     6344
2023     7334
2024    10039
2025    15332
dtype: int64

In [10]:
### records inside the Saemangeum seawall
### (the boundary is in EPSG:5186 now, so change it back to lat/lon first)
saemangeum_records_gdf = gbif_gdf.sjoin(
    saemangeum_gdf.to_crs(gbif_gdf.crs), how = 'inner', predicate = 'within')

### number of records inside the seawall in each year
saemangeum_records_gdf.groupby('year').size()

year
2002     18
2003      6
2004      7
2006     27
2008      1
2011      1
2013      1
2014      5
2015     10
2016      4
2017    122
2018     13
2019      5
2020     22
2021      2
2022      8
2023     43
2024     43
2025    199
dtype: int64

In [11]:
### check the two study sites on a satellite map
geum_gdf = gpd.read_file('../data/geum.geojson')
hvplot.extension('bokeh')
site_map = (
    saemangeum_gdf.to_crs(epsg = 4326).hvplot(
        geo = True, tiles = 'EsriImagery', fill_color = None, line_color = 'red',
        line_width = 2, label = 'Saemangeum',
        frame_height = 400, frame_width = 350,
        title = 'Study sites: Saemangeum and the Geum Estuary')
    *
    geum_gdf.hvplot(
        geo = True, fill_color = None, line_color = 'yellow',
        line_width = 2, label = 'Geum Estuary')
).opts(legend_position = 'right')
site_map

:Overlay
   .WMTS.I                :WMTS   [Longitude,Latitude]
   .Polygons.Saemangeum   :Polygons   [Longitude,Latitude]
   .Polygons.Geum_Estuary :Polygons   [Longitude,Latitude]

In [12]:
### count shorebird records in each site for each period
### share = site records / all records in Korea in the same period (%)
geum_records_gdf = gbif_gdf.sjoin(geum_gdf, how = 'inner', predicate = 'within')

periods = {
    '2000-2005': [2000, 2005],
    '2006-2012': [2006, 2012],
    '2013-2019': [2013, 2019],
    '2020-2025': [2020, 2025]}

share_list = []
for period, years in periods.items():
    korea_count = ((gbif_gdf.year >= years[0]) & (gbif_gdf.year <= years[1])).sum()
    saemangeum_count = ((saemangeum_records_gdf.year >= years[0]) & (saemangeum_records_gdf.year <= years[1])).sum()
    geum_count = ((geum_records_gdf.year >= years[0]) & (geum_records_gdf.year <= years[1])).sum()
    share_list.append({
        'Period': period,
        'Korea records': korea_count,
        'Saemangeum records': saemangeum_count,
        'Geum Estuary records': geum_count,
        'Saemangeum': round(saemangeum_count / korea_count * 100, 2),
        'Geum Estuary': round(geum_count / korea_count * 100, 2)})

share_df = pd.DataFrame(share_list).set_index('Period')
share_df

,Korea records,Saemangeum records,Geum Estuary records,Saemangeum,Geum Estuary
Period,,,,,
2000-2005,1159,31,89,2.67,7.68
2006-2012,1789,29,406,1.62,22.69
2013-2019,9304,160,2012,1.72,21.63
2020-2025,46694,317,5590,0.68,11.97


In [13]:
### Figure 3: share of Korean shorebird records in each site
hvplot.extension('bokeh')
share_plot = share_df[['Saemangeum', 'Geum Estuary']].hvplot.line(
    title = 'Share of shorebird records in Korea (GBIF, Scolopacidae)',
    xlabel = 'Period',
    ylabel = 'Share of records in Korea (%)',
    frame_width = 500,
    frame_height = 300,
    legend = 'top_right',
    group_label = 'Site')
share_plot

:NdOverlay   [Site]
   :Curve   [Period]   (value)

In [14]:
### save the figures as html files for the blog post
hvplot.save(site_map, '../img/saemangeum-site-map.html')
hvplot.save(landcover_maps, '../img/saemangeum-landcover-maps.html')
hvplot.save(landcover_bar, '../img/saemangeum-landcover-area.html')
hvplot.save(share_plot, '../img/saemangeum-shorebird-share.html')